<a href="https://colab.research.google.com/github/sobaginspoorti/Secure-String-Matching1/blob/main/secure_string_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ✅ Install required packages
!pip install cryptography bcrypt ipywidgets

# ---------------- Imports ----------------
import time
import bcrypt
import matplotlib.pyplot as plt
from cryptography.fernet import Fernet
from IPython.display import HTML, display
import random
import ipywidgets as widgets

# ---------------- Styled Output ----------------
def styled_message(message, bg_color="lightblue", text_color="black"):
    html = f"<div style='background-color:{bg_color}; color:{text_color}; padding:10px; border-radius:5px; font-weight:bold;'>{message}</div>"
    display(HTML(html))

# ---------------- Authentication ----------------
stored_username = "admin"
stored_password_hash = bcrypt.hashpw("1234".encode(), bcrypt.gensalt())

def authenticate():
    username = input("Enter Username: ")
    password = input("Enter Password: ").encode()
    if username == stored_username and bcrypt.checkpw(password, stored_password_hash):
        styled_message("✅ Authentication Successful!", bg_color="green", text_color="white")
        return True
    else:
        styled_message("❌ Authentication Failed!", bg_color="red", text_color="white")
        return False

# ---------------- Two-Factor Authentication ----------------
def two_factor_auth():
    otp = str(random.randint(100000, 999999))
    styled_message(f"Your OTP is: {otp}", bg_color="purple", text_color="white")
    entered = input("Enter OTP: ")
    if entered == otp:
        styled_message("✅ 2FA Successful!", bg_color="green", text_color="white")
        return True
    else:
        styled_message("❌ 2FA Failed!", bg_color="red", text_color="white")
        return False

# ---------------- AES Encryption / Decryption ----------------
def generate_key():
    return Fernet.generate_key()

def encrypt(text, key):
    f = Fernet(key)
    return f.encrypt(text.encode()).decode()

def decrypt(cipher_text, key):
    f = Fernet(key)
    return f.decrypt(cipher_text.encode()).decode()

# ---------------- KMP Algorithm ----------------
def compute_lps(pattern):
    lps = [0] * len(pattern)
    length = 0
    i = 1
    while i < len(pattern):
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]
            else:
                lps[i] = 0
                i += 1
    return lps

def kmp_search(text, pattern):
    lps = compute_lps(pattern)
    i = j = 0
    positions = []
    while i < len(text):
        if pattern[j] == text[i]:
            i += 1
            j += 1
        if j == len(pattern):
            positions.append(i - j)
            j = lps[j - 1]
        elif i < len(text) and pattern[j] != text[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return positions

# ---------------- Naive Search ----------------
def naive_search(text, pattern):
    positions = []
    for i in range(len(text) - len(pattern) + 1):
        if text[i:i+len(pattern)] == pattern:
            positions.append(i)
    return positions

# ---------------- Boyer-Moore Search ----------------
def bad_char_table(pattern):
    table = [-1] * 256
    for i in range(len(pattern)):
        table[ord(pattern[i])] = i
    return table

def boyer_moore_search(text, pattern):
    m = len(pattern)
    n = len(text)
    table = bad_char_table(pattern)
    positions = []
    s = 0
    while s <= n - m:
        j = m - 1
        while j >= 0 and pattern[j] == text[s + j]:
            j -= 1
        if j < 0:
            positions.append(s)
            s += (m - table[ord(text[s + m])] if s + m < n else 1)
        else:
            s += max(1, j - table[ord(text[s + j])])
    return positions

# ---------------- Highlight Matches ----------------
def highlight_matches(text, positions, pattern):
    highlighted = ""
    last_index = 0
    for pos in positions:
        highlighted += text[last_index:pos] + f"<span style='background-color:yellow; color:black; font-weight:bold;'>{text[pos:pos+len(pattern)]}</span>"
        last_index = pos + len(pattern)
    highlighted += text[last_index:]
    return highlighted

# ---------------- Performance Visualization ----------------
def plot_performance(times):
    plt.bar(times.keys(), times.values(), color=["red","green","blue"])
    plt.ylabel("Execution Time (seconds)")
    plt.title("Naive vs KMP vs Boyer-Moore Search Performance")
    plt.show()

# ---------------- Main Program ----------------
if authenticate() and two_factor_auth():
    choice = input("Do you want to input text or upload file? (text/file): ").strip().lower()
    if choice == "file":
        from google.colab import files
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        with open(file_name, "r") as f:
            text = f.read()
    else:
        text = input("Enter Text: ")

    pattern = input("Enter Pattern: ")

    # AES Encryption / Decryption
    key = generate_key()
    encrypted_text = encrypt(text, key)
    styled_message("🔒 Encrypted Text: " + encrypted_text, bg_color="lightyellow", text_color="black")

    decrypted_text = decrypt(encrypted_text, key)
    styled_message("🔓 Decrypted Text: " + decrypted_text, bg_color="lightcyan", text_color="black")

    # Case-insensitive search
    text_lower = decrypted_text.lower()
    pattern_lower = pattern.lower()

    # Dropdown for algorithm choice
    algo_dropdown = widgets.Dropdown(
        options=["Naive", "KMP", "Boyer-Moore"],
        value="KMP",
        description="Algorithm:"
    )
    display(algo_dropdown)

    def run_search(algo_choice):
        times = {}
        positions = []

        if algo_choice == "Naive":
            start_time = time.time()
            positions = naive_search(text_lower, pattern_lower)
            times["Naive"] = time.time() - start_time
        elif algo_choice == "KMP":
            start_time = time.time()
            positions = kmp_search(text_lower, pattern_lower)
            times["KMP"] = time.time() - start_time
        else:
            start_time = time.time()
            positions = boyer_moore_search(text_lower, pattern_lower)
            times["Boyer-Moore"] = time.time() - start_time

        # Results
        if positions:
            styled_message(f"Pattern Found at Positions: {positions}", bg_color="lightgreen", text_color="black")
            styled_message(f"Match Count: {len(positions)}", bg_color="lightblue", text_color="black")
            display(HTML(highlight_matches(decrypted_text, positions, pattern)))
        else:
            styled_message("Pattern Not Found", bg_color="orange", text_color="black")

        # Performance
        for k,v in times.items():
            styled_message(f"{k} Search Time: {v:.6f} seconds", bg_color="lightpink", text_color="black")

        plot_performance(times)
        styled_message("✅ Search Completed Securely", bg_color="purple", text_color="white")

    # ✅ Bind dropdown to function
    widgets.interact(run_search, algo_choice=algo_dropdown)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.8 MB/s eta 0:00:00
Enter Username: admin
Enter Password: 1234


Enter OTP: 224956


Do you want to input text or upload file? (text/file): text
Enter Text: welcome to hak-a-league
Enter Pattern: a


Dropdown(description='Algorithm:', index=1, options=('Naive', 'KMP', 'Boyer-Moore'), value='KMP')

interactive(children=(Dropdown(description='Algorithm:', index=1, options=('Naive', 'KMP', 'Boyer-Moore'), val…